# AML - SAIGE-GENE SET-BASED ANALYSIS - 2000 RANDOM VARIANTS 
This script generates a list of 2,000 randomly selected variants for variance ratio estimation in SAIGE-GENE set-based analysis. Multiple variance ratios must be estimated based on different minor allele count (MAC) categories when –isCateVarianceRatio=TRUE.
This differs from Step 1 in SAIGE for single-variant tests, where only a single variance ratio is estimated. By default, two variance ratios are estimated for the categories 10 ≤ MAC < 20 and MAC ≥ 20. 

In [ ]:
# Load packages 
import os
import numpy as np
import pandas as pd

In [ ]:
# Set environment varaible for workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")
# Check environment variable for workspace bucket 
my_bucket

In [ ]:
# Delete and create new directory for microarray counts
!rm -rf microarray_counts
!mkdir -p microarray_counts

In [ ]:
# Download microarray counts from workspace bucket 
!gsutil cp {my_bucket}/data/saige_step01/microarray_counts/EUR-like_microarray.acount microarray_counts

In [ ]:
# Loop for filtering for autosomal chromosomes 
for pop in ["EUR"]: 
    print("===================================")
    print(f"Processing population {pop} ....")
    
    # Read the microarray counts data frame into working environemnt 
    microarray = pd.read_csv(
        f"microarray_counts/{pop}-like_microarray.acount",
        sep="\t",
        dtype={"#CHROM": "string"})
    
    # Filter for autosomes  
    microarray = microarray[
        microarray["#CHROM"]
        .astype(str)
        .isin([str(i) for i in range(1, 23)])
    ]
    
    # Check for the presence of Y chomosome 
    if microarray["#CHROM"].isin(["Y"]).any():
        print("Presence of Y chromosome ....")
        break 
    
    # Check for the presence of X chomosome 
    if microarray["#CHROM"].isin(["X"]).any():
        print("Presence of X chromosome ....")
        break 
    
    # Check that 22 chromosomes are present 
    if len(microarray["#CHROM"].unique()) != 22:
        print("Total number of chromosomes is not 22 ....")
        break 
    
    # Check that chromosomes 1-22 are present 
    if not np.isin(microarray["#CHROM"].astype(int), range(1, 23)).all():
        print("Missing chromosome 1-22 in microarray counts ....")
        break 
        
    # Convert column to chromosome 
    microarray["#CHROM"] = pd.to_numeric(microarray["#CHROM"], errors="coerce")
    # Save microarray counts 
    microarray.to_csv(f"microarray_counts/{pop}-like_autosomes_microarray.acount", 
                      sep ='\t',
                      index = None)
    # Check counts for each chromosome 
    microarray_check = microarray["#CHROM"].value_counts().reset_index().sort_values(by="#CHROM")
    print("===================================")
    print(microarray_check[["#CHROM", "count"]].to_string(index=False))
    print("===================================")

In [ ]:
%%bash 
# Path for autosomal microarray counts 
path=microarray_counts
# Loop for selecting 2000 randomally selected markers 
for pop in EUR; do 
    echo "======================================================"
    echo "Processing population $pop"
     echo "======================================================"
    echo "Processing ${pop}_autosomes_microarray.acount ...."
    cat <(tail -n +2 $path/${pop}-like_autosomes_microarray.acount | \
    awk '(($6-$5) < 20 && ($6-$5) >= 10) || ($5 < 20 && $5 >= 10) {print $2}' | \
    shuf -n 1000) <(tail -n +2 $path/${pop}-like_autosomes_microarray.acount | \
    awk ' $5 >= 20 && ($6-$5)>= 20 {print $2}' | shuf -n 1000) > $path/${pop}.markerid.list

done

In [ ]:
%%bash
# Check the 2000 randomally selected markers 
echo "========================================="
wc -l microarray_counts/EUR.markerid.list
echo "========================================="
head microarray_counts/EUR.markerid.list
echo "========================================="

In [ ]:
# Copy marker list to workspace bucket 
!gsutil cp microarray_counts/EUR.markerid.list {my_bucket}/data/saige_step01/mark_list/ 

In [ ]:
# Check that marker list has been successfully copied to workspace bucket 
!gsutil cat {my_bucket}/data/saige_step01/mark_list/EUR.markerid.list | wc -l  